In [72]:
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold , train_test_split, GridSearchCV,ParameterGrid
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import (OrdinalEncoder , OneHotEncoder )
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import (LinearRegression , Ridge , Lasso)
from sklearn.ensemble import (RandomForestRegressor , GradientBoostingRegressor)
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

### 1.Importer et couper le jeux de données en train et test

In [32]:
df = pd.read_csv("data_cleaning/annonces.csv" , sep=";")
df.head()



,Loyer_TTC,Surface,Charges,Pièces,label_eco,kWh,kgCO2,Quartiers
0,756,30,20.0,2.0,D,204.0,6.0,Quartiers Centre
1,1770,117,50.0,5.0,E,241.0,52.0,Quartiers Centre
2,481,22,15.0,1.0,D,241.0,9.0,Quartiers Centre
3,1190,64,135.0,3.0,C,118.0,9.0,Quartiers Centre
4,1260,68,160.0,3.0,E,280.0,61.0,Quartiers Centre


In [33]:
X=df.drop(columns=["Loyer_TTC"])
Y = df["Loyer_TTC"]
x_train , x_test , y_train , y_test = train_test_split(X,Y,test_size=0.25 ,random_state=42)

In [51]:
#Imputer
set(df["label_eco"])



{'B+', 'C', 'D', 'E', 'F'}

Avec .ravel() :


x_train["label_eco"] = encoder.fit_transform(x_train[["label_eco"]]).ravel()

x_test["label_eco"]  = encoder.transform(x_test[["label_eco"]]).ravel()

Ou avec des doubles crochets des deux côtés, sans .ravel() :

x_train[["label_eco"]] = encoder.fit_transform(x_train[["label_eco"]])

x_test[["label_eco"]]  = encoder.transform(x_test[["label_eco"]])

In [50]:
df["label_eco"].unique()

<StringArray>
['D', 'E', 'C', 'B+', 'F']
Length: 5, dtype: str

In [62]:
def creer_pipeline(methode , avec_pca=False): #exemple methode = LogisticRegression()
    ordre = [['B+', 'C', 'D', 'E', 'F']]
    pipe_encoder = Pipeline(steps=[
        ("encoder" , OrdinalEncoder(categories=ordre))
    ])

    pipe_one_hot = Pipeline(steps=[
        ("encoder" , OneHotEncoder())
    ])
    pipe_pca = Pipeline(steps=[
        ("scale" , StandardScaler()),
        ("PCA" , PCA() )
    ])
    preprocesser = ColumnTransformer(transformers=[
        ("ord" , pipe_encoder ,["label_eco"] ) ,
        ("one_hot" , pipe_one_hot , ["Quartiers"] )
    ], remainder="passthrough")
    preprocesser_avec_pca = ColumnTransformer(transformers=[
        ("ord" , pipe_encoder ,["label_eco"]) ,
        ("one_hot" , pipe_one_hot , ["Quartiers"]) ,
        ("pca" , pipe_pca ,["Surface" , "Charges" , "kWh" , "kgCO2"])
    ], remainder="passthrough")
    if avec_pca :
        return Pipeline(steps=[
            ("preprocesser" , preprocesser_avec_pca),
            ("modele" ,methode )
        ])
    return Pipeline(steps=[
               ("preprocesser" , preprocesser),
               ("modele" ,methode )
           ])

In [74]:
alphas = np.logspace(-3, 3, 13)
k= np.arange(1,100,1)
p = np.arange(0.05,0.5 , 0.05)
models = {
    "KNN"  :(creer_pipeline(KNeighborsRegressor() , avec_pca=True) ,
                          {"modele__n_neighbors" : k }),
    "gradient boosting" : (creer_pipeline(GradientBoostingRegressor()) ,
                          {"modele__loss" :['squared_error', 'absolute_error', 'huber'] ,
                           "modele__learning_rate" : p  ,
                           "modele__n_estimators":[50,75 ,100]}) ,
    "random forest regression" :(creer_pipeline(RandomForestRegressor()) ,
                             {"modele__n_estimators":[50,75 ,100], "modele__max_depth":[None , 5 , 10 ,15 ,20] }) ,
    "Régression Lasso"  :(creer_pipeline(Lasso()) ,
                          {"modele__alpha" : alphas }),
    "Régression Ridge"  :(creer_pipeline(Ridge()) ,
                            {"modele__alpha" : alphas }) 
}

In [75]:
from math import inf
cv = KFold(5)
metric ={"MSE" : "neg_mean_squared_error" ,
         "MAE" : "neg_mean_absolute_error" ,
         "R2" : "r2"
         }
meilleur = None
best_score = inf
best_parametre = None
best_estimateur = None
for nom_methode , (pipe_line , grid_param) in models.items() :
    print( f"{nom_methode}")
    grid_search = GridSearchCV(estimator=pipe_line , param_grid=grid_param,cv =cv ,
                               scoring=metric , refit ="MSE" , n_jobs=-1)
    grid_search.fit(x_train,y_train)
    i = grid_search.best_index_
    res = grid_search.cv_results_
    print("Best parametres :", grid_search.best_params_)
    print("MSE :", -res["mean_test_MSE"][i])
    print("MAE :", -res["mean_test_MAE"][i])
    print("R2  :",  res["mean_test_R2"][i])
    if -res["mean_test_MSE"][i] <=best_score :
        best_score = -res["mean_test_MSE"][i]
        meilleur = nom_methode 
        best_parametre =grid_search.best_params_
        best_estimateur = grid_search.best_estimator_
print(meilleur )
print(best_parametre)


KNN
Best parametres : {'modele__n_neighbors': np.int64(1)}
MSE : 59373.10641258318
MAE : 73.68983666061706
R2  : 0.7582660113598197
gradient boosting
Best parametres : {'modele__learning_rate': np.float64(0.35000000000000003), 'modele__loss': 'squared_error', 'modele__n_estimators': 75}
MSE : 45102.287056935005
MAE : 65.55831444498673
R2  : 0.8583856017238871
random forest regression
Best parametres : {'modele__max_depth': 15, 'modele__n_estimators': 100}
MSE : 50968.31249763812
MAE : 87.20594861487658
R2  : 0.8450548790705217
Régression Lasso
Best parametres : {'modele__alpha': np.float64(10.0)}
MSE : 60681.86266229748
MAE : 139.5619378573015
R2  : 0.6583917882824843
Régression Ridge
Best parametres : {'modele__alpha': np.float64(31.622776601683793)}
MSE : 62423.19525713658
MAE : 140.13237368260894
R2  : 0.6450626763421685
gradient boosting
{'modele__learning_rate': np.float64(0.35000000000000003), 'modele__loss': 'squared_error', 'modele__n_estimators': 75}


In [76]:
    # ou meilleurs["Random Forest"]

y_pred = best_estimateur.predict(x_test)
print("MAE  test :", mean_absolute_error(y_test, y_pred))
print("RMSE test :", np.sqrt(mean_squared_error(y_test, y_pred)))
print("R²   test :", r2_score(y_test, y_pred))
print("R²  train :", best_estimateur.score(x_train, y_train))

MAE  test : 59.527366787516286
RMSE test : 150.40579484655387
R²   test : 0.9081117435141947
R²  train : 0.9982692822324748
